In [117]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import lightgbm as lgb
import optuna

import warnings
warnings.filterwarnings("ignore")

In [73]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df_train. head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [74]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [75]:
for df in [df_train, df_test]:
    df['Group'] = df.PassengerId.str.split('_').str[0]
    df[['Desk', 'Num', 'Side']] = df.Cabin.str.split('/', expand=True)
    df['Num'] = pd.to_numeric(df["Num"],errors="coerce")
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['Total'] = df[spend_cols].sum(axis=1)
    df['No_cash'] = (df["Total"]==0).astype("int8")
    df['Ages'] = pd.cut(df.Age, bins=5, labels=False)
    df["DeskFreq"] = df.groupby("Desk")["Desk"].transform("count")

In [76]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 22 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
 14  Group         8693 non-null   object 
 15  Desk          8494 non-null   object 
 16  Num           8494 non-null   float64
 17  Side          8494 non-null   object 
 18  Total         8693 non-null 

In [77]:
for df in [df_train,df_test]:

    df["Age"] = df["Age"].fillna(df["Age"].median())

    spend_cols = ["RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]

    for col in spend_cols:
        df[col] = df[col].fillna(0)

    for col in ["CryoSleep","VIP"]:
        df[col] = df[col].fillna(False)
        df[col] = df[col].astype("int8")

In [78]:
cat_cols = ["HomePlanet","Destination","Desk","Side"]

for col in cat_cols:

    df_train[col] = df_train[col].fillna(df_train[col].mode()[0])
    df_test[col] = df_test[col].fillna(df_test[col].mode()[0])

df_train["Num"] = df_train["Num"].fillna(df_train["Num"].median())
df_test["Num"] = df_test["Num"].fillna(df_test["Num"].median())

In [79]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 22 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8693 non-null   object 
 2   CryoSleep     8693 non-null   int8   
 3   Cabin         8494 non-null   object 
 4   Destination   8693 non-null   object 
 5   Age           8693 non-null   float64
 6   VIP           8693 non-null   int8   
 7   RoomService   8693 non-null   float64
 8   FoodCourt     8693 non-null   float64
 9   ShoppingMall  8693 non-null   float64
 10  Spa           8693 non-null   float64
 11  VRDeck        8693 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
 14  Group         8693 non-null   object 
 15  Desk          8693 non-null   object 
 16  Num           8693 non-null   float64
 17  Side          8693 non-null   object 
 18  Total         8693 non-null 

In [80]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 22 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8693 non-null   object 
 2   CryoSleep     8693 non-null   int8   
 3   Cabin         8494 non-null   object 
 4   Destination   8693 non-null   object 
 5   Age           8693 non-null   float64
 6   VIP           8693 non-null   int8   
 7   RoomService   8693 non-null   float64
 8   FoodCourt     8693 non-null   float64
 9   ShoppingMall  8693 non-null   float64
 10  Spa           8693 non-null   float64
 11  VRDeck        8693 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
 14  Group         8693 non-null   object 
 15  Desk          8693 non-null   object 
 16  Num           8693 non-null   float64
 17  Side          8693 non-null   object 
 18  Total         8693 non-null 

In [123]:
X_train = df_train.drop(['PassengerId', 'Cabin', 'Name', 'Group', 'Transported'], axis = 1)
y_train = df_train.Transported
test = df_train.drop(['PassengerId', 'Cabin', 'Name', 'Group', 'Transported'], axis = 1)

In [124]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   HomePlanet    8693 non-null   object 
 1   CryoSleep     8693 non-null   int8   
 2   Destination   8693 non-null   object 
 3   Age           8693 non-null   float64
 4   VIP           8693 non-null   int8   
 5   RoomService   8693 non-null   float64
 6   FoodCourt     8693 non-null   float64
 7   ShoppingMall  8693 non-null   float64
 8   Spa           8693 non-null   float64
 9   VRDeck        8693 non-null   float64
 10  Desk          8693 non-null   object 
 11  Num           8693 non-null   float64
 12  Side          8693 non-null   object 
 13  Total         8693 non-null   float64
 14  No_cash       8693 non-null   int8   
 15  Ages          8514 non-null   float64
 16  DeskFreq      8494 non-null   float64
dtypes: float64(10), int8(3), object(4)
memory usage: 976.4+ KB


In [125]:
X_train = pd.get_dummies(X_train)
test = pd.get_dummies(test)

In [102]:
rf = LGBMClassifier(n_jobs=5, random_state = 5)

In [105]:
parameters = {
    'boosting_type': ['gbdt', 'dart'],
    'num_leaves': range(5, 31, 5),
    'max_depth': range(3, 25, 2),
    'learning_rate': [0.1, 0.2, 0.5, 0.7],
    'n_estimators': range(10, 151, 10),
    'min_child_samples ': range(2, 30, 5),
}

In [108]:
clf = RandomizedSearchCV(rf, parameters, cv = 3, n_jobs=5, verbose=2, n_iter=1000)

In [118]:
def objective(trial):

    params = {

        "n_estimators":trial.suggest_int("n_estimators",400,1000),

        "learning_rate":trial.suggest_float("lr",0.01,0.08),

        "num_leaves":trial.suggest_int("leaves",20,120),

        "max_depth":trial.suggest_int("depth",3,10),

        "subsample":trial.suggest_float("subsample",0.7,1),

        "colsample_bytree":trial.suggest_float("colsample",0.7,1),

        "verbosity":-1

    }

    model = lgb.LGBMClassifier(**params)

    skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

    scores=[]

    for tr_idx,val_idx in skf.split(X_train,y_train):

        Xtr,Xval = X_train.iloc[tr_idx],X_train.iloc[val_idx]

        ytr,yval = y_train.iloc[tr_idx],y_train.iloc[val_idx]

        model.fit(Xtr,ytr)

        pred = model.predict(Xval)

        scores.append(accuracy_score(yval,pred))

    return np.mean(scores)

study = optuna.create_study(direction="maximize")

study.optimize(objective,n_trials=15)

best_params = study.best_params

[I 2026-03-19 12:06:12,854] A new study created in memory with name: no-name-7fb7f1c1-1f0e-4b10-9886-df0edeb8387c
[I 2026-03-19 12:06:13,912] Trial 0 finished with value: 0.8106521941964979 and parameters: {'n_estimators': 477, 'lr': 0.04503250185189969, 'leaves': 28, 'depth': 5, 'subsample': 0.8727726825174872, 'colsample': 0.9627771818056402}. Best is trial 0 with value: 0.8106521941964979.
[I 2026-03-19 12:06:14,586] Trial 1 finished with value: 0.8074310262567735 and parameters: {'n_estimators': 400, 'lr': 0.020000752551397206, 'leaves': 81, 'depth': 4, 'subsample': 0.7475044529949282, 'colsample': 0.8627159272590523}. Best is trial 0 with value: 0.8106521941964979.
[I 2026-03-19 12:06:16,475] Trial 2 finished with value: 0.8083512276078935 and parameters: {'n_estimators': 901, 'lr': 0.07360439952172204, 'leaves': 85, 'depth': 5, 'subsample': 0.9768058870260352, 'colsample': 0.7227052012607164}. Best is trial 0 with value: 0.8106521941964979.
[I 2026-03-19 12:06:18,581] Trial 3 fin

In [120]:
lgb_model = lgb.LGBMClassifier(**best_params)

In [126]:
skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

test_preds = np.zeros(len(test))

scores=[]

for tr_idx,val_idx in skf.split(X_train,y_train):

    Xtr,Xval = X_train.iloc[tr_idx],X_train.iloc[val_idx]

    ytr,yval = y_train.iloc[tr_idx],y_train.iloc[val_idx]

    lgb_model.fit(Xtr,ytr)

    pred = lgb_model.predict(Xval)

    scores.append(accuracy_score(yval,pred))

    test_preds += lgb_model.predict(test)

print("CV Accuracy:",np.mean(scores))

CV Accuracy: 0.8011037651759441
